In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Read the file from Google Drive
df = pd.read_csv('/content/drive/MyDrive/combined_spam_dataset.csv')

# Drop rows with missing values in the text and label columns
# (Note: If your dataset has different column names, replace 'text' and 'label' with your actual column names)
df = df.dropna(subset=['text', 'label'])

X = df['text']
y = df['label']

# Split the data: 80% for Training and 20% for Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Data Split Successful! Train size:", len(X_train), "| Test size:", len(X_test))

Data Split Successful! Train size: 9454 | Test size: 2364


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("--- Training Logistic Regression Model ---")

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Model Training
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

# Model Predictions
y_pred_lr = lr_model.predict(X_test_tfidf)

# Display Evaluation Metrics
print("\n=== Logistic Regression Results ===")
print("Accuracy :", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall   :", recall_score(y_test, y_pred_lr))
print("F1-Score :", f1_score(y_test, y_pred_lr))

--- Training Logistic Regression Model ---

=== Logistic Regression Results ===
Accuracy : 0.9576988155668359
Precision: 0.9841772151898734
Recall   : 0.8735955056179775
F1-Score : 0.9255952380952381


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

print("--- Training LSTM Model ---")

vocab_size = 10000
max_length = 100

# Tokenization
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_padded = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_length, padding='post')
X_test_padded = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_length, padding='post')

# LSTM Architecture
lstm_model = Sequential([
    Embedding(vocab_size, 64, input_length=max_length),
    LSTM(64),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Training
lstm_model.fit(X_train_padded, y_train, epochs=5, batch_size=32, validation_split=0.1)

# Evaluation
y_pred_lstm_prob = lstm_model.predict(X_test_padded)
y_pred_lstm = (y_pred_lstm_prob > 0.5).astype(int)

print("\n=== LSTM Results ===")
print("Accuracy :", accuracy_score(y_test, y_pred_lstm))
print("Precision:", precision_score(y_test, y_pred_lstm))
print("Recall   :", recall_score(y_test, y_pred_lstm))
print("F1-Score :", f1_score(y_test, y_pred_lstm))

--- Training LSTM Model ---
Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


266/266 ━━━━━━━━━━━━━━━━━━━━ 22s 70ms/step - accuracy: 0.8217 - loss: 0.4640 - val_accuracy: 0.8214 - val_loss: 0.4525
Epoch 2/5
266/266 ━━━━━━━━━━━━━━━━━━━━ 18s 69ms/step - accuracy: 0.8333 - loss: 0.4302 - val_accuracy: 0.8266 - val_loss: 0.4706
Epoch 3/5
266/266 ━━━━━━━━━━━━━━━━━━━━ 19s 73ms/step - accuracy: 0.8335 - loss: 0.4115 - val_accuracy: 0.8975 - val_loss: 0.3095
Epoch 4/5
266/266 ━━━━━━━━━━━━━━━━━━━━ 20s 74ms/step - accuracy: 0.9201 - loss: 0.2255 - val_accuracy: 0.9397 - val_loss: 0.1962
Epoch 5/5
266/266 ━━━━━━━━━━━━━━━━━━━━ 18s 68ms/step - accuracy: 0.9620 - loss: 0.1420 - val_accuracy: 0.9577 - val_loss: 0.1522
74/74 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step

=== LSTM Results ===
Accuracy : 0.9581218274111675
Precision: 0.9622926093514329
Recall   : 0.8960674157303371
F1-Score : 0.928
